Nama Lengkap : Nazarul Bagus Riyadi

NIM : 240401010229

Kelas : IF404

---

# Praktikum Pertemuan 3: Data Cleaning — Missing Values, Outlier & Ekstraksi Data


**Import library**
- `pandas` : mengolah data tabel
- `numpy` : operasi numerik
- `requests` & `json_normalize` : ekstraksi data dari API


In [1]:
# PIPELINE DATA CLEANING - HOUSING DATASET
# Nazarul Bagus Riyadi (240401010229)

import pandas as pd
import numpy as np
import requests
from pandas import json_normalize

print('Library berhasil diimport')



Library berhasil diimport


**Eksplorasi awal** — tujuan:
- Melihat ukuran dataset
- Memeriksa tipe data dan statistik deskriptif
- Mengidentifikasi missing values sebelum dibersihkan


In [2]:
url = 'https://raw.githubusercontent.com/arulriyadi/data-science-source/main/housing_dirty.csv'
df = pd.read_csv(url)
print('Shape awal:', df.shape)
print(df.info())
print(df.describe().round(2))
print('\nMissing values:')
print(df.isnull().sum())


Shape awal: (130, 7)
<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    str    
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    str    
dtypes: float64(3), int64(2), str(2)
memory usage: 7.2 KB
None
           id  luas_m2   harga_juta   kamar  tahun_bangun
count  130.00   112.00       113.00  120.00        130.00
mean    65.50   267.63    885632.51    3.43       2062.64
std     37.67   885.66   9407144.37    1.78        701.68
min      1.00   -50.00      -500.00    1.00       1890.00
25%     33.25    87.05       345.00    2.00       1991.25
50%     65.50   193.80       655.00    4.00       2002.00
75%     97.75   280.68       9

**STEP 1 — Hapus duplikat**
- `drop_duplicates()` menghapus baris yang sama persis
- `inplace=True` langsung mengubah DataFrame asli


In [3]:
df.drop_duplicates(inplace=True)
print('Shape setelah hapus duplikat:', df.shape)


Shape setelah hapus duplikat: (130, 7)


**STEP 2 & 3 — Normalisasi string & imputasi**
- Kota seperti "jakarta", "JAKARTA" diseragamkan dengan `.str.title()`
- Kondisi diseragamkan dengan `.str.lower()`
- Missing numerik diisi **median** (lebih tahan outlier)
- Missing kategorik diisi **modus**


In [4]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print('Missing setelah imputasi:')
print(df.isnull().sum())


Missing setelah imputasi:
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


**STEP 4 — Outlier dengan IQR Fence**
- Q1 = kuartil 25%, Q3 = kuartil 75%
- IQR = Q3 − Q1
- `clip()` membatasi nilai di luar batas Q1 − 1.5×IQR dan Q3 + 1.5×IQR


In [5]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)
    print(f'{col}: batas [{lower:.2f}, {upper:.2f}]')


harga_juta: batas [-422.75, 1719.25]
luas_m2: batas [-145.22, 512.97]
tahun_bangun: batas [1960.50, 2042.50]


**STEP 5 — Validasi & ekspor**
- Pastikan tidak ada missing values dan duplikat
- Simpan hasil ke `housing_clean.csv`


In [6]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing value!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Dataset sudah bersih!')
df.to_csv('housing_clean.csv', index=False)
print('housing_clean.csv berhasil disimpan!')
print('Shape akhir:', df.shape)


Dataset sudah bersih!
housing_clean.csv berhasil disimpan!
Shape akhir: (130, 7)


**STEP 6 — Ekstraksi data dari REST API**
- Mengambil data dari JSONPlaceholder (`/users`)
- Mengubah respons JSON menjadi DataFrame dengan `json_normalize()`


In [7]:
url = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(url, timeout=10)
print('Status code:', response.status_code)

if response.status_code == 200:
    df_api = json_normalize(response.json())
    print(df_api.head())
    print('\nShape data API:', df_api.shape)
else:
    print('Error:', response.status_code)


Status code: 200
   id  ...                            company.bs
0   1  ...           harness real-time e-markets
1   2  ...      synergize scalable supply-chains
2   3  ...       e-enable strategic applications
3   4  ...  transition cutting-edge web services
4   5  ...      revolutionize end-to-end systems

[5 rows x 15 columns]

Shape data API: (10, 15)


## Kesimpulan

### Apa yang Dipelajari
Pada pertemuan ini saya mempelajari **data cleaning** menggunakan dataset `housing_dirty.csv`:
- Eksplorasi awal: shape, info, describe, dan missing values
- Menghapus duplikat dengan `drop_duplicates()`
- Normalisasi string dan imputasi missing values (median/modus)
- Menangani outlier dengan metode **IQR Fence** dan `clip()`
- Validasi dataset lalu mengekspor ke `housing_clean.csv`
- Mengambil data dari **REST API** publik dan mengonversinya ke DataFrame

### Temuan Utama
- Dataset awal memiliki missing values di kolom `luas_m2`, `harga_juta`, dan `kamar`
- Terdapat inkonsistensi format pada kolom `kota` dan `kondisi` yang perlu dinormalisasi
- Outlier pada harga dan luas bangunan dapat mendistorsi analisis jika tidak ditangani
- API JSONPlaceholder berhasil diakses dan dikonversi menjadi DataFrame siap analisis

### Keterbatasan
- Pipeline ini belum menangani semua jenis data kotor (misalnya duplikat semantik atau typo nama)
- Imputasi median/modus adalah pendekatan sederhana; dataset nyata mungkin memerlukan KNN Imputer
